# Step 06 — Four search algorithms on London road network


In [ ]:
import sys
import subprocess
from pathlib import Path


current = Path.cwd().resolve()

project_root = next(
    (
        p for p in [current, *current.parents]
        if (p / "pyproject.toml").exists()
        and (p / "src" / "routing").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not find AiLeason project root. "
        "Open JupyterLab from the AiLeason folder."
    )

print("Project root:", project_root)
print("Python kernel:", sys.executable)

# Install into the exact Python interpreter used by this notebook
subprocess.check_call([
    sys.executable,
    "-m", "pip",
    "install",
    "-e", str(project_root)
])

print("Editable installation completed.")

Project root: /Users/mojtabashabani/wiser/AiWithPython/AiLeason
Python kernel: /Users/mojtabashabani/wiser/mary/.venv/bin/python
Obtaining file:///Users/mojtabashabani/wiser/AiWithPython/AiLeason
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for london-traffic-routing (pyproject.toml): started
  Building editable for london-traffic-routing (pyproject.toml): finished with status 'done'
  Created wheel for london-traffic-routing: filename=london_traffic_routing-0.1.0-0.editable-py3-none-any.whl size=1408 sha256=ac8c49a4f980431


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [19]:
import sys
import pandas as pd
from IPython.display import display

from routing.config import ORIGIN, ORIGINS, DESTINATION
from routing.map_loader import load_road_graph, nearest_drive_node
from routing.map_view import interactive_map
from routing.traffic_pipeline import collect_traffic_snapshot, make_routing_graph
from routing.algorithms import bfs, dfs, greedy_best_first, astar, run_all_algorithms
from routing.route_view import add_route_overlay

print("Python kernel:", sys.executable)
RADIUS_M = 2000
G = load_road_graph(radius_m=RADIUS_M)
print("OSM graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "directed edges")
assert G.is_directed() and G.is_multigraph()

ModuleNotFoundError: No module named 'routing'

## 2 — Prepare one shared, safe offline snapshot

In [20]:


snapshot, summary = collect_traffic_snapshot(G, None, max_requests=0)
assert summary["requests_made"] == 0
assert summary["usable_observed_edges"] == 0

measured_time = make_routing_graph(snapshot, objective="time", scenario="measured")
measured_distance = make_routing_graph(snapshot, objective="distance", scenario="measured")
free_flow_time = make_routing_graph(snapshot, objective="time", scenario="free_flow")

print("API requests:", summary["requests_made"])
print("Cost attribute (time):", measured_time.graph["routing_weight"])
print("Cost attribute (distance):", measured_distance.graph["routing_weight"])
print("NOTICE: offline measured travel times are IMPUTED, not actual traffic observations.")

NameError: name 'collect_traffic_snapshot' is not defined

## 3 — Choose your origin by double-clicking on the map



In [21]:
route_widget, route_state = interactive_map(
    G, origin=ORIGIN, destination=DESTINATION, traffic_graph=snapshot,
    max_snap_distance_m=350.0,
)
display(route_widget)

NameError: name 'interactive_map' is not defined

## 4 — Capture the selected nodes

In [ ]:
source = route_state["origin_node"]
goal = route_state["destination_node"]
print("Origin node:", source)
print("Destination node:", goal)
print("Last double-click:", route_state["last_click_latlon"])
assert source in G and goal in G



NameError: name 'route_state' is not defined

## 5 — Run all four algorithms (time objective)

In [ ]:
results_time = run_all_algorithms(measured_time, source, goal)

cols = ["algorithm", "found", "hops", "distance_m", "travel_time_s",
        "objective_cost", "imputed_route_edges", "expanded_nodes",
        "max_frontier_size", "runtime_ms"]
rows = [{k: result[k] for k in cols} for result in results_time.values()]
display(pd.DataFrame(rows).round(3))

if not results_time["A*"]["found"]:
    print("No directed path between these selected nodes. Choose another road for the origin.")
else:
    print("A* time-optimal path:", results_time["A*"]["path_edges"][:8], "...")

NameError: name 'run_all_algorithms' is not defined

## 6 — Overlay the four paths on the SAME working JupyterLab map



In [ ]:
for name, result in results_time.items():
    if result["found"]:
        add_route_overlay(route_widget, G, result, name=f"{name} | time")
        print(name, "route drawn; edges:", len(result["path_edges"]))
    else:
        print(name, "no route available")
print("Routes were added to the map displayed above.")

NameError: name 'results_time' is not defined

## 7 — Change objective: distance (same source, goal, and snapshot)

In [ ]:
results_distance = run_all_algorithms(measured_distance, source, goal)
display(pd.DataFrame([
    {"algorithm": name, "found": r["found"], "distance_m": r["distance_m"],
     "travel_time_s": r["travel_time_s"], "expanded_nodes": r["expanded_nodes"],
     "runtime_ms": r["runtime_ms"]}
    for name, r in results_distance.items()
]).round(3))

if results_distance["A*"]["found"]:
    print("A* minimum distance:", results_distance["A*"]["distance_m"], "m")
    

NameError: name 'run_all_algorithms' is not defined

## 8 — Free-flow time scenario

In [ ]:
results_free = run_all_algorithms(free_flow_time, source, goal)
for name, result in results_free.items():
    print(name, "found:", result["found"],
          "free-flow seconds:", result["travel_time_s"])


NameError: name 'run_all_algorithms' is not defined

## 9 — Road closure scenario: remove an edge from the A* route



In [ ]:
from routing.traffic_pipeline import make_routing_graph

best_route = results_time["A*"]
if best_route["found"] and best_route["path_edges"]:
    chosen_closure = best_route["path_edges"][len(best_route["path_edges"]) // 2]
    closure_graph = make_routing_graph(
        snapshot, objective="time", scenario="closure",
        imposed_closure=chosen_closure,
    )
    assert not closure_graph.has_edge(*chosen_closure)
    closure_results = run_all_algorithms(closure_graph, source, goal)
    print("Imposed closure:", chosen_closure)
    display(pd.DataFrame([
        {"algorithm": name, "found": r["found"],
         "travel_time_s": r["travel_time_s"], "hops": r["hops"]}
        for name, r in closure_results.items()
    ]).round(3))
else:
    print("No nonempty A* path available for a closure demonstration.")

ModuleNotFoundError: No module named 'routing'

## 10 — Automatic smoke checks (NO NetworkX shortest-path used in implementations)

In [ ]:
for results in (results_time, results_distance, results_free):
    for name, result in results.items():
        if result["found"]:
            assert result["path_nodes"][0] == source
            assert result["path_nodes"][-1] == goal
            assert all(G.has_edge(*edge) for edge in result["path_edges"])
            assert len(result["path_edges"]) == len(result["path_nodes"]) - 1
print("PASS: Four algorithms have valid route structures across three graph objectives/scenarios")
print("Reminder: realistic traffic-aware comparison needs a quality-checked live snapshot.")

NameError: name 'results_time' is not defined